# 03 — Factories avec `factory_boy`

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- créer des **factories** pour générer des objets de test réalistes
- utiliser `Faker` pour des données aléatoires crédibles
- maîtriser `LazyFunction`, `LazyAttribute`, `SubFactory`
- intégrer les factories avec `pytest`

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- pytest, fixtures
- dataclasses, pydantic
- mocking

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- rien de spécifique

## Plan

1. Pourquoi des factories
2. Installation et `factory.Factory`
3. `Faker` et `LazyFunction`
4. `LazyAttribute`
5. `SubFactory`
6. Intégration pytest
7. Synthèse
8. Exercices

---

## 1. Pourquoi des factories

Dans les tests, on a besoin d'**objets réalistes** mais variés. Construire chaque objet à la main est fastidieux et fragile. Une **factory** génère des instances avec des valeurs par défaut raisonnables qu'on override au besoin.

---

## 2. Installation et `factory.Factory`

In [ ]:
# uv add factory-boy (ou pip install factory-boy)
import factory
from dataclasses import dataclass


@dataclass
class Utilisateur:
    nom: str
    email: str
    age: int


class UtilisateurFactory(factory.Factory):
    class Meta:
        model = Utilisateur

    nom = 'Alice'
    email = factory.LazyAttribute(lambda o: f'{o.nom.lower()}@test.fr')
    age = 30


In [ ]:
UtilisateurFactory()


In [ ]:
UtilisateurFactory(nom='Bob', age=25)


In [ ]:
UtilisateurFactory.build_batch(3)


---

## 3. `Faker` et `LazyFunction`

Faker génère des données réalistes (noms, emails, adresses, dates).

In [ ]:
class UtilisateurFakerFactory(factory.Factory):
    class Meta:
        model = Utilisateur

    nom = factory.Faker('name', locale='fr_FR')
    email = factory.Faker('email')
    age = factory.LazyFunction(lambda: __import__('random').randint(18, 70))


In [ ]:
UtilisateurFakerFactory()


In [ ]:
UtilisateurFakerFactory()


---

## 4. `LazyAttribute`

Calcule une valeur **à partir d'autres attributs** de la factory.

In [ ]:
class ProduitFactory(factory.Factory):
    class Meta:
        model = type('Produit', (), {'__init__': lambda self, **kw: self.__dict__.update(kw), '__repr__': lambda self: repr(self.__dict__)})

    nom = factory.Faker('word')
    prix_ht = factory.LazyFunction(lambda: round(__import__('random').uniform(1, 100), 2))
    prix_ttc = factory.LazyAttribute(lambda o: round(o.prix_ht * 1.2, 2))


In [ ]:
p = ProduitFactory()


In [ ]:
p.prix_ht, p.prix_ttc


---

## 5. `SubFactory`

Quand un objet contient un autre objet.

In [ ]:
@dataclass
class Reservation:
    utilisateur: Utilisateur
    salle: str
    creneau: str


class ReservationFactory(factory.Factory):
    class Meta:
        model = Reservation

    utilisateur = factory.SubFactory(UtilisateurFakerFactory)
    salle = factory.Iterator(['Mars', 'Venus', 'Io'])
    creneau = factory.Faker('time', pattern='%Hh%M')


In [ ]:
ReservationFactory()


In [ ]:
ReservationFactory(utilisateur__nom='Alice')


---

## 6. Intégration pytest

Les factories s'utilisent directement dans les tests, souvent via une fixture.

In [ ]:
import pytest

@pytest.fixture
def utilisateur() -> Utilisateur:
    return UtilisateurFakerFactory()

def test_email_contient_at(utilisateur):
    assert '@' in utilisateur.email


---

## Synthèse

| Outil | Rôle |
|---|---|
| `factory.Factory` | Classe de base |
| `Faker(provider)` | Données réalistes |
| `LazyFunction(callable)` | Valeur calculée sans contexte |
| `LazyAttribute(lambda o: ...)` | Valeur calculée depuis l'objet |
| `SubFactory(F)` | Sous-objet |
| `build_batch(n)` | N instances |


### Règles à retenir

1. **Factories pour les objets de test**, pas de dicts construits à la main.
2. **Faker pour des données réalistes** : ça attrape des bugs qu'on ne voit pas avec 'test'.
3. **SubFactory pour les relations** : ça compose naturellement.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — Factory basique *(facile)*

Écrire `LivreFactory` pour `@dataclass Livre(titre, auteur, pages)` avec Faker.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Factories", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import factory
from dataclasses import dataclass

@dataclass
class Livre:
    titre: str
    auteur: str
    pages: int

class LivreFactory(factory.Factory):
    class Meta:
        model = Livre
    titre = factory.Faker('sentence', nb_words=3)
    auteur = factory.Faker('name')
    pages = factory.LazyFunction(lambda: __import__('random').randint(50, 500))

for _ in range(3):
    print(LivreFactory())
```

</details>

### Exercice 2 — SubFactory *(moyen)*

Écrire `EquipeFactory` pour `Equipe(nom, leader: Utilisateur)` avec `SubFactory`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Factories", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import factory
from dataclasses import dataclass

@dataclass
class Equipe:
    nom: str
    leader: Utilisateur

class EquipeFactory(factory.Factory):
    class Meta:
        model = Equipe
    nom = factory.Faker('company')
    leader = factory.SubFactory(UtilisateurFakerFactory)

print(EquipeFactory())
```

</details>

### Exercice 3 — Batch + assertions *(difficile)*

Générer 10 réservations avec `ReservationFactory.build_batch(10)`. Vérifier qu'aucune n'a de créneau vide.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Factories", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
batch = ReservationFactory.build_batch(10)
assert all(r.creneau for r in batch)
print(f'{len(batch)} réservations valides')
```

</details>

---

## Ressources externes

### Documentation officielle
- [factory_boy docs](https://factoryboy.readthedocs.io/)
- [Faker providers](https://faker.readthedocs.io/en/master/providers.html)